# BLAST-Enhanced Pretrained Model Segmentation for Oral Cancer Histopathology

**Objective:** Compare 5 pretrained CNN encoders **with vs without BLAST** enhancement for histopathological image segmentation using Leave-One-Image-Out cross-validation.

| Mode | Description |
|------|-------------|
| **Without BLAST** | Frozen pretrained encoder + trainable decoder (10 epochs) |
| **With BLAST** | BLAST k-NN prediction map fed as 4th channel to encoder+decoder (10 epochs) |

**Models:** ResNet50, VGG16, DenseNet121, MobileNetV2, EfficientNetB0  
**Classes:** Normal (0), Dysplasia (1), Carcinoma (2)

In [ ]:
# ── Setup & Imports ──
import os, warnings, time, gc
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# GPU setup (Colab-compatible)
nvidia_base = "/usr/local/lib/python3.10/dist-packages/nvidia"
nvidia_libs = [
    f"{nvidia_base}/cuda_runtime/lib", f"{nvidia_base}/cudnn/lib",
    f"{nvidia_base}/cublas/lib", f"{nvidia_base}/cufft/lib",
    f"{nvidia_base}/curand/lib", f"{nvidia_base}/cusolver/lib",
    f"{nvidia_base}/cusparse/lib", f"{nvidia_base}/nvjitlink/lib",
    "/usr/lib/wsl/lib",
]
os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_libs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, jaccard_score
)
from sklearn.preprocessing import normalize

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import (
    ResNet50, VGG16, DenseNet121, MobileNetV2, EfficientNetB0
)
from tensorflow.keras.applications import (
    resnet50, vgg16, densenet, mobilenet_v2, efficientnet
)

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print(f'TensorFlow: {tf.__version__}  |  GPUs: {gpus}')

In [ ]:
# ── Configuration ──
SEED       = 42
IMG_SIZE   = 256
PATCH_SIZE = 64
NUM_IMAGES = 10
NUM_CLASSES = 3
EPOCHS     = 10
TOP_K      = 5

PATCHES_PER_SIDE  = IMG_SIZE // PATCH_SIZE   # 4
PATCHES_PER_IMAGE = PATCHES_PER_SIDE ** 2    # 16

CLASS_NAMES  = ['Normal', 'Dysplasia', 'Carcinoma']
CLASS_COLORS = {0: [144, 238, 144], 1: [255, 255, 102], 2: [255, 99, 71]}

np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR = 'outputs'
DATA_DIR   = 'data/synthetic_images'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Config: {IMG_SIZE}x{IMG_SIZE} images, {PATCH_SIZE}x{PATCH_SIZE} patches, '
      f'{NUM_IMAGES} images, {EPOCHS} epochs')

## Synthetic Data Generation
Generate 10 synthetic H&E-stained oral tissue images with pixel-level segmentation masks.

In [ ]:
def draw_nuclei(img, mask, region_mask, cls, rng):
    """Draw nuclei into a region based on tissue class."""
    ys, xs = np.where(region_mask)
    if len(ys) == 0:
        return
    area = len(ys)

    if cls == 0:      # Normal
        n_nuclei = max(3, area // 900)
        radius_range = (2, 4)
        color_base = np.array([80, 40, 120])
    elif cls == 1:    # Dysplasia
        n_nuclei = max(5, area // 500)
        radius_range = (3, 6)
        color_base = np.array([60, 20, 100])
    else:             # Carcinoma
        n_nuclei = max(8, area // 250)
        radius_range = (3, 8)
        color_base = np.array([40, 10, 80])

    for _ in range(n_nuclei):
        idx = rng.randint(0, len(ys))
        cy, cx = ys[idx], xs[idx]
        r = rng.randint(radius_range[0], radius_range[1] + 1)
        color_var = rng.randint(-15, 16, size=3)
        color = np.clip(color_base + color_var, 0, 255).tolist()
        cv2.circle(img, (cx, cy), r, color, -1)
        if cls == 2 and rng.random() > 0.5:
            offset = rng.randint(-3, 4, size=2)
            cv2.circle(img, (cx + offset[0], cy + offset[1]), r - 1, color, -1)


def generate_tissue_image(img_id, rng):
    """Generate one synthetic H&E image with mixed-class segmentation mask."""
    img  = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

    bg_colors = {
        0: np.array([230, 200, 210]),
        1: np.array([215, 180, 200]),
        2: np.array([200, 160, 185]),
    }

    n_regions = rng.randint(2, 5)
    centers = rng.randint(30, IMG_SIZE - 30, size=(n_regions, 2))

    if img_id < 3:
        region_classes = rng.choice([0, 0, 1], size=n_regions); region_classes[0] = 0
    elif img_id < 6:
        region_classes = rng.choice([0, 1, 2], size=n_regions); region_classes[0] = 1
    else:
        region_classes = rng.choice([1, 2, 2], size=n_regions); region_classes[0] = 2

    yy, xx = np.mgrid[0:IMG_SIZE, 0:IMG_SIZE]
    dist_maps = np.stack([(yy - cy)**2 + (xx - cx)**2 for cy, cx in centers], axis=-1)
    nearest = np.argmin(dist_maps, axis=-1)

    for r_idx in range(n_regions):
        region_mask = nearest == r_idx
        cls = region_classes[r_idx]
        mask[region_mask] = cls
        bg = bg_colors[cls].astype(np.float64)
        noise = rng.normal(0, 5, size=(IMG_SIZE, IMG_SIZE, 3))
        for c in range(3):
            img[:, :, c][region_mask] = np.clip(
                bg[c] + noise[:, :, c][region_mask], 0, 255
            ).astype(np.uint8)
        draw_nuclei(img, mask, region_mask, cls, rng)

    img = cv2.GaussianBlur(img, (3, 3), 0.7)
    return img, mask


# Generate all images
images, masks = [], []
for i in range(NUM_IMAGES):
    rng = np.random.RandomState(SEED + i)
    img, msk = generate_tissue_image(i, rng)
    images.append(img)
    masks.append(msk)
    cv2.imwrite(os.path.join(DATA_DIR, f'image_{i:02d}.png'), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(DATA_DIR, f'mask_{i:02d}.png'), msk)

images_np = np.array(images, dtype=np.float32)
masks_np  = np.array(masks,  dtype=np.uint8)
print(f'Generated {NUM_IMAGES} images: {images_np.shape}, masks: {masks_np.shape}')

In [ ]:
# Visualize sample images with ground truth masks
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for row, img_id in enumerate([0, 4, 8]):
    axes[row, 0].imshow(images[img_id])
    axes[row, 0].set_title(f'Image {img_id}')
    axes[row, 0].axis('off')

    color_mask = np.zeros((*masks[img_id].shape, 3), dtype=np.uint8)
    for c, clr in CLASS_COLORS.items():
        color_mask[masks[img_id] == c] = clr
    axes[row, 1].imshow(color_mask)
    axes[row, 1].set_title(f'GT Mask {img_id}')
    axes[row, 1].axis('off')

    overlay = (images[img_id].astype(np.float32) * 0.6 +
               color_mask.astype(np.float32) * 0.4).astype(np.uint8)
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f'Overlay {img_id}')
    axes[row, 2].axis('off')

legend_patches = [mpatches.Patch(color=np.array(c)/255, label=n)
                  for n, c in zip(CLASS_NAMES, CLASS_COLORS.values())]
fig.legend(handles=legend_patches, loc='upper center', ncol=3, fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_images_masks.png'), dpi=120, bbox_inches='tight')
plt.show()

## Pretrained Models
Define 5 ImageNet-pretrained encoders and a lightweight segmentation decoder.

In [ ]:
PRETRAINED_MODELS = {
    'ResNet50':       (ResNet50,       resnet50.preprocess_input),
    'VGG16':          (VGG16,          vgg16.preprocess_input),
    'DenseNet121':    (DenseNet121,    densenet.preprocess_input),
    'MobileNetV2':    (MobileNetV2,    mobilenet_v2.preprocess_input),
    'EfficientNetB0': (EfficientNetB0, efficientnet.preprocess_input),
}


def build_segmentation_model(model_class, preprocess_fn,
                             input_shape=(IMG_SIZE, IMG_SIZE, 3),
                             num_classes=NUM_CLASSES):
    """Frozen pretrained encoder + trainable decoder."""
    inputs = layers.Input(shape=input_shape)
    preprocessed = layers.Lambda(lambda x: preprocess_fn(x))(inputs)

    encoder = model_class(weights='imagenet', include_top=False,
                          input_shape=input_shape)
    encoder.trainable = False
    encoded = encoder(preprocessed)

    x = layers.Conv2D(64, 1, activation='relu')(encoded)
    x = layers.Resizing(IMG_SIZE, IMG_SIZE, interpolation='bilinear')(x)
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
    x = layers.Conv2D(num_classes, 1, activation='softmax')(x)

    model = Model(inputs, x)
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


def build_blast_segmentation_model(model_class, preprocess_fn,
                                   input_shape=(IMG_SIZE, IMG_SIZE, 4),
                                   num_classes=NUM_CLASSES):
    """4-channel (RGB + BLAST map) encoder + decoder.
    Projects 4 channels to 3 with a trainable 1x1 conv before the frozen encoder."""
    inputs = layers.Input(shape=input_shape)

    projected = layers.Conv2D(3, 1, activation='relu', name='channel_proj')(inputs)
    preprocessed = layers.Lambda(lambda x: preprocess_fn(x))(projected)

    encoder = model_class(weights='imagenet', include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    encoder.trainable = False
    encoded = encoder(preprocessed)

    x = layers.Conv2D(64, 1, activation='relu')(encoded)
    x = layers.Resizing(IMG_SIZE, IMG_SIZE, interpolation='bilinear')(x)
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
    x = layers.Conv2D(num_classes, 1, activation='softmax')(x)

    model = Model(inputs, x)
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


def build_patch_feature_extractor(model_class, input_shape=(PATCH_SIZE, PATCH_SIZE, 3)):
    """Frozen pretrained model for patch-level feature extraction."""
    base = model_class(weights='imagenet', include_top=False,
                       input_shape=input_shape, pooling='avg')
    base.trainable = False
    return base


# Show embedding dimensions
print(f'{"Model":<18} {"Embedding Dim":>14}')
print('-' * 34)
for name, (cls, _) in PRETRAINED_MODELS.items():
    ext = build_patch_feature_extractor(cls)
    print(f'{name:<18} {ext.output_shape[-1]:>14}')
    del ext
    tf.keras.backend.clear_session()

In [ ]:
def compute_segmentation_metrics(true_masks, pred_seg_maps, method_name):
    """Compute pixel-level segmentation metrics across all images."""
    all_true, all_pred = [], []
    for img_id in range(NUM_IMAGES):
        all_true.append(true_masks[img_id].flatten())
        all_pred.append(pred_seg_maps[img_id].flatten())

    all_true = np.concatenate(all_true)
    all_pred = np.concatenate(all_pred)

    pixel_acc = accuracy_score(all_true, all_pred)
    mean_iou  = jaccard_score(all_true, all_pred, average='macro')

    dice_scores = []
    for c in range(NUM_CLASSES):
        gt_c   = (all_true == c)
        pred_c = (all_pred == c)
        intersection = (gt_c & pred_c).sum()
        dice = 2 * intersection / (gt_c.sum() + pred_c.sum() + 1e-10)
        dice_scores.append(dice)
    mean_dice = np.mean(dice_scores)

    prec = precision_score(all_true, all_pred, average='macro', zero_division=0)
    rec  = recall_score(all_true, all_pred, average='macro', zero_division=0)
    f1   = f1_score(all_true, all_pred, average='macro', zero_division=0)

    return {
        'Method': method_name,
        'Pixel Accuracy': pixel_acc,
        'Mean IoU': mean_iou,
        'Mean Dice': mean_dice,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'Dice_per_class': dice_scores,
        'all_true': all_true,
        'all_pred': all_pred,
    }


print('Metrics function defined.')

## Mode 1: Without BLAST
Train each pretrained encoder (frozen) + lightweight decoder for 10 epochs using Leave-One-Image-Out CV.

In [ ]:
print('MODE 1: Pretrained Models - Direct Segmentation (No BLAST)')
print('=' * 60)

direct_seg_maps = {}
direct_metrics_list = []

for model_name, (model_class, preprocess_fn) in PRETRAINED_MODELS.items():
    print(f'\n-- {model_name} (Direct) --')
    seg_maps = {}
    t0 = time.time()

    for img_id in range(NUM_IMAGES):
        train_idx = [i for i in range(NUM_IMAGES) if i != img_id]
        X_train = images_np[train_idx]
        y_train = masks_np[train_idx][..., np.newaxis]
        X_test  = images_np[img_id:img_id+1]

        tf.keras.backend.clear_session()
        gc.collect()

        fold_model = build_segmentation_model(model_class, preprocess_fn)
        fold_model.fit(X_train, y_train, epochs=EPOCHS, batch_size=2, verbose=0)

        pred_prob = fold_model.predict(X_test, verbose=0)[0]
        seg_maps[img_id] = np.argmax(pred_prob, axis=-1).astype(np.uint8)

        del fold_model
        tf.keras.backend.clear_session()
        gc.collect()

    elapsed = time.time() - t0
    direct_seg_maps[model_name] = seg_maps

    m = compute_segmentation_metrics(masks, seg_maps, f'{model_name} (Direct)')
    direct_metrics_list.append(
        {k: v for k, v in m.items() if k not in ['Dice_per_class', 'all_true', 'all_pred']}
    )
    print(f'  Done in {elapsed:.0f}s | Acc: {m["Pixel Accuracy"]:.4f}  '
          f'IoU: {m["Mean IoU"]:.4f}  Dice: {m["Mean Dice"]:.4f}  F1: {m["F1 Score"]:.4f}')

print('\n-- Direct Segmentation Summary --')
pd.DataFrame(direct_metrics_list).set_index('Method').round(4)

## BLAST Technique
BLAST (Basic Local Alignment Search Tool) adapted for histopathology: extract patch embeddings with pretrained encoders, perform k-NN matching against a reference database, and generate a BLAST prediction map as an additional input channel.

In [ ]:
class BLASTImageDatabase:
    """BLAST-like database for histopathological patch matching."""

    def __init__(self, metric='cosine', top_k=TOP_K):
        self.metric = metric
        self.top_k = top_k
        self.db_features = None
        self.db_labels = None

    def build_database(self, features, labels):
        self.db_features = normalize(features.copy())
        self.db_labels = labels.copy()

    def query(self, query_feature):
        query_norm = normalize(query_feature.reshape(1, -1))
        if self.metric == 'cosine':
            sims = (query_norm @ self.db_features.T).flatten()
        else:
            dists = np.linalg.norm(self.db_features - query_norm, axis=1)
            sims = 1.0 / (1.0 + dists)

        top_idx = np.argsort(sims)[::-1][:self.top_k]
        top_labels = self.db_labels[top_idx]
        weights = np.maximum(sims[top_idx], 1e-10)

        class_scores = np.zeros(NUM_CLASSES)
        for lbl, w in zip(top_labels, weights):
            class_scores[lbl] += w

        return np.argmax(class_scores)

    def segment_image(self, patch_features):
        """Query all patches and assemble full-resolution segmentation map."""
        preds = [self.query(f) for f in patch_features]
        pred_grid = np.array(preds).reshape(PATCHES_PER_SIDE, PATCHES_PER_SIDE)
        seg_map = np.kron(pred_grid, np.ones((PATCH_SIZE, PATCH_SIZE), dtype=np.uint8))
        return seg_map


def extract_all_patches(images_list):
    """Extract non-overlapping patches and labels from all images."""
    all_patches, all_labels, img_indices = [], [], []

    for img_id, (img, msk) in enumerate(zip(images_list, masks)):
        for r in range(PATCHES_PER_SIDE):
            for c in range(PATCHES_PER_SIDE):
                y0, x0 = r * PATCH_SIZE, c * PATCH_SIZE
                patch = img[y0:y0+PATCH_SIZE, x0:x0+PATCH_SIZE]
                label_patch = msk[y0:y0+PATCH_SIZE, x0:x0+PATCH_SIZE]
                all_patches.append(patch)
                vals, counts = np.unique(label_patch, return_counts=True)
                all_labels.append(vals[np.argmax(counts)])
                img_indices.append(img_id)

    return np.array(all_patches), np.array(all_labels), np.array(img_indices)


def generate_blast_maps(model_class, preprocess_fn, model_name):
    """Generate BLAST prediction maps for all images using LOIO-CV."""
    all_patches, all_labels, img_indices = extract_all_patches(images)

    tf.keras.backend.clear_session()
    gc.collect()
    extractor = build_patch_feature_extractor(model_class)
    patches_float = all_patches.astype(np.float32)
    patches_prep = np.array([preprocess_fn(p.copy()) for p in patches_float])
    embeddings = extractor.predict(patches_prep, batch_size=16, verbose=0)
    del extractor, patches_float, patches_prep
    tf.keras.backend.clear_session()
    gc.collect()

    blast_maps = {}
    for img_id in range(NUM_IMAGES):
        test_mask  = img_indices == img_id
        train_mask = ~test_mask
        db = BLASTImageDatabase(metric='cosine', top_k=TOP_K)
        db.build_database(embeddings[train_mask], all_labels[train_mask])
        blast_maps[img_id] = db.segment_image(embeddings[test_mask])

    return blast_maps


print('BLAST database class and helper functions defined.')

## Mode 2: With BLAST Enhancement
For each model: generate BLAST prediction map, concatenate as 4th channel (RGB + BLAST), then train encoder+decoder for 10 epochs with LOIO-CV.

In [ ]:
print('MODE 2: Pretrained Models + BLAST Enhancement')
print('=' * 60)

blast_seg_maps = {}
blast_metrics_list = []

for model_name, (model_class, preprocess_fn) in PRETRAINED_MODELS.items():
    print(f'\n-- {model_name} + BLAST --')
    t0 = time.time()

    # Step 1: Generate BLAST maps for all images
    print('  Generating BLAST maps...')
    blast_maps = generate_blast_maps(model_class, preprocess_fn, model_name)

    # Step 2: Create 4-channel inputs (RGB + BLAST map scaled to 0-255)
    blast_channel_all = np.zeros((NUM_IMAGES, IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
    for img_id in range(NUM_IMAGES):
        blast_channel_all[img_id, :, :, 0] = (
            blast_maps[img_id].astype(np.float32) * (255.0 / max(NUM_CLASSES - 1, 1))
        )

    images_4ch = np.concatenate([images_np, blast_channel_all], axis=-1)

    # Step 3: LOIO-CV training with 4-channel input
    seg_maps = {}
    print('  Training with BLAST channel...')
    for img_id in range(NUM_IMAGES):
        train_idx = [i for i in range(NUM_IMAGES) if i != img_id]
        X_train = images_4ch[train_idx]
        y_train = masks_np[train_idx][..., np.newaxis]
        X_test  = images_4ch[img_id:img_id+1]

        tf.keras.backend.clear_session()
        gc.collect()

        fold_model = build_blast_segmentation_model(model_class, preprocess_fn)
        fold_model.fit(X_train, y_train, epochs=EPOCHS, batch_size=2, verbose=0)

        pred_prob = fold_model.predict(X_test, verbose=0)[0]
        seg_maps[img_id] = np.argmax(pred_prob, axis=-1).astype(np.uint8)

        del fold_model
        tf.keras.backend.clear_session()
        gc.collect()

    elapsed = time.time() - t0
    blast_seg_maps[model_name] = seg_maps

    m = compute_segmentation_metrics(masks, seg_maps, f'{model_name} + BLAST')
    blast_metrics_list.append(
        {k: v for k, v in m.items() if k not in ['Dice_per_class', 'all_true', 'all_pred']}
    )
    print(f'  Done in {elapsed:.0f}s | Acc: {m["Pixel Accuracy"]:.4f}  '
          f'IoU: {m["Mean IoU"]:.4f}  Dice: {m["Mean Dice"]:.4f}  F1: {m["F1 Score"]:.4f}')

print('\n-- BLAST Enhancement Summary --')
pd.DataFrame(blast_metrics_list).set_index('Method').round(4)

## Results Comparison
Compare all 10 method variants (5 models x 2 modes) side by side.

In [ ]:
# Build unified comparison table
all_metrics = []
all_metrics_full = {}

for model_name in PRETRAINED_MODELS.keys():
    m_direct = compute_segmentation_metrics(
        masks, direct_seg_maps[model_name], f'{model_name} (Direct)')
    all_metrics.append(m_direct)
    all_metrics_full[f'{model_name} (Direct)'] = m_direct

    m_blast = compute_segmentation_metrics(
        masks, blast_seg_maps[model_name], f'{model_name} + BLAST')
    all_metrics.append(m_blast)
    all_metrics_full[f'{model_name} + BLAST'] = m_blast

metrics_df = pd.DataFrame([
    {k: v for k, v in m.items() if k not in ['Dice_per_class', 'all_true', 'all_pred']}
    for m in all_metrics
]).set_index('Method').round(4)

print('=' * 90)
print('COMPARISON: With BLAST vs Without BLAST (all 10 methods)')
print('=' * 90)
metrics_df

In [ ]:
# Grouped bar chart: BLAST vs Direct for each model
model_names = list(PRETRAINED_MODELS.keys())
metric_cols = ['Pixel Accuracy', 'Mean IoU', 'Mean Dice', 'F1 Score']

fig, axes = plt.subplots(1, len(metric_cols), figsize=(20, 6))
for ax_idx, metric_name in enumerate(metric_cols):
    blast_vals  = [metrics_df.loc[f'{m} + BLAST', metric_name] for m in model_names]
    direct_vals = [metrics_df.loc[f'{m} (Direct)', metric_name] for m in model_names]

    x = np.arange(len(model_names))
    w = 0.35
    axes[ax_idx].bar(x - w/2, blast_vals,  w, label='With BLAST',    color='#4C72B0')
    axes[ax_idx].bar(x + w/2, direct_vals, w, label='Without BLAST', color='#C44E52')
    axes[ax_idx].set_xticks(x)
    axes[ax_idx].set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
    axes[ax_idx].set_title(metric_name, fontsize=12)
    axes[ax_idx].set_ylim(0, 1.05)
    axes[ax_idx].legend(fontsize=8)

plt.suptitle('Pretrained Models: With BLAST vs Without BLAST', fontsize=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'blast_vs_direct_bars.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices for all 10 methods
n_methods = len(all_metrics)
n_cols = 5
n_rows = 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))

for idx, m in enumerate(all_metrics):
    row, col = divmod(idx, n_cols)
    cm = confusion_matrix(m['all_true'], m['all_pred'], labels=[0, 1, 2])
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-10)

    ax = axes[row, col]
    im = ax.imshow(cm_norm, vmin=0, vmax=1, cmap='Blues')
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                    fontsize=9, color='white' if cm_norm[i,j] > 0.5 else 'black')
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, fontsize=7, rotation=30)
    ax.set_yticklabels(CLASS_NAMES, fontsize=7)
    ax.set_title(m['Method'], fontsize=9)

plt.suptitle('Confusion Matrices (Normalized)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Visual segmentation comparison for sample images
sample_imgs = [0, 4, 8]
model_names_vis = list(PRETRAINED_MODELS.keys())

fig, axes = plt.subplots(len(sample_imgs), 2 + 2 * len(model_names_vis),
                         figsize=(4 * (2 + 2 * len(model_names_vis)),
                                  4 * len(sample_imgs)))

for row, img_id in enumerate(sample_imgs):
    axes[row, 0].imshow(images[img_id])
    axes[row, 0].set_title('Original' if row == 0 else '', fontsize=9)
    axes[row, 0].axis('off')

    gt_color = np.zeros((*masks[img_id].shape, 3), dtype=np.uint8)
    for c, clr in CLASS_COLORS.items():
        gt_color[masks[img_id] == c] = clr
    axes[row, 1].imshow(gt_color)
    axes[row, 1].set_title('GT' if row == 0 else '', fontsize=9)
    axes[row, 1].axis('off')

    col = 2
    for mname in model_names_vis:
        seg_d = direct_seg_maps[mname][img_id]
        seg_d_color = np.zeros((*seg_d.shape, 3), dtype=np.uint8)
        for c, clr in CLASS_COLORS.items():
            seg_d_color[seg_d == c] = clr
        axes[row, col].imshow(seg_d_color)
        axes[row, col].set_title(f'{mname}\nDirect' if row == 0 else '', fontsize=8)
        axes[row, col].axis('off')
        col += 1

        seg_b = blast_seg_maps[mname][img_id]
        seg_b_color = np.zeros((*seg_b.shape, 3), dtype=np.uint8)
        for c, clr in CLASS_COLORS.items():
            seg_b_color[seg_b == c] = clr
        axes[row, col].imshow(seg_b_color)
        axes[row, col].set_title(f'{mname}\n+BLAST' if row == 0 else '', fontsize=8)
        axes[row, col].axis('off')
        col += 1

legend_patches = [mpatches.Patch(color=np.array(c)/255, label=n)
                  for n, c in zip(CLASS_NAMES, CLASS_COLORS.values())]
fig.legend(handles=legend_patches, loc='upper center', ncol=3, fontsize=11)
plt.suptitle('Segmentation Maps: Direct vs BLAST Enhanced', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'segmentation_maps_comparison.png'),
            dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Save metrics to CSV
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'segmentation_metrics.csv'))
print(f'Metrics saved to {OUTPUT_DIR}/segmentation_metrics.csv')
metrics_df